**Investments: Theory and Data Analysis**, Bates, Boyer, and Fletcher


# Example Chapter 11: Optimal Active Portfolio
In this example we estimate the optimal active tilt towards the zero-cost portfolio that goes long decile 1 short-term reversal portfolio and short decile 10 short-term reversal portfolio from the Ken French Data Library.


### Imports and Setup

In [ ]:
# Import packages
%pip install -q --upgrade farms
import farms as fm
import pandas as pd
import numpy as np

# Establish display options when printing DataFrames
pd.set_option('display.max_columns', None)   # Show all columns without truncation
pd.set_option('display.width', 1000)   # Set the display width so output stays on one line
pd.set_option("display.max_rows", 20) # Force truncation if DataFrame has more than 20 rows

### Load in Data Using `farms.load_ken_french_data()`
For full parameter details, see the [`farms` API documentation](https://github.com/boyerb/farms/blob/main/docs/api.md#load_ken_french_data).

`load_ken_french_data()`

**Required Inputs**

- `data_type` — Selects the portfolio granularity. Potential values are `"deciles"` and `"quintiles"`. Example: `data_type='deciles'`.
- `strategy` — Selects the firm characteristic used to sort stocks. Example: `strategy='shorttermreversal'`.

**Optional Inputs**

- `include_factors` — Default is `None`. Adds factor returns to the portfolio DataFrame. Potential values are `None`, `"market"`, `"ff3"`, and `"ff5"`. Example: `include_factors='market'`.
- `frequency` — Default is `"monthly"`. Selects the observation frequency when it is available for the strategy. Potential values are `"daily"`, `"weekly"`, and `"monthly"`. Example: `frequency='monthly'`.

**Outputs**

- `DataFrame` — Portfolio returns indexed by observation date, with decimal return columns and the requested market-factor columns. Example: `df=fm.load_ken_french_data(...)`.

In [ ]:
# Get monthly short-term reversal deciles and market factors
df = fm.load_ken_french_data(
    data_type='deciles',
    strategy='shorttermreversal',
    include_factors='market',
)
print(df)

### Define Inputs
In the code cell below we define the covariance matrix and the vector of expected returns.  Here you can enter your own covriance matrix and vector of expected returns as desired, with any number of securities. You can also load these inputs from your account on Google Drive, or your own GitHub repo.   

In [ ]:
df = df.copy()  # avoid SettingWithCopyWarning

# Define the zero-cost portfolio
df['dec1_dec10'] = df['dec1'] - df['dec10']

# Estimate Expected excess returns
expected_returns = df[['mkt-rf', 'dec1_dec10']].mean().to_numpy()

# Estimate the covariance matrix
covariance_matrix = df[['mkt-rf', 'dec1_dec10']].cov().to_numpy()
print(expected_returns)
print(covariance_matrix)
# Note that the covariance matrix here is defined using the excess market return.
# Subtracting off the risk-free rate, which is close to constant, has very little
# effect on the covariance matrix.  In some applications is is common to define
# the covariance matrix using the excess market return.

### Define the Optimal Active Portfolio
For full parameter details, see the [`farms` portfolio API documentation](https://github.com/boyerb/farms/blob/main/docs/api.md#factor_tilt_portfolio).

`factor_tilt_portfolio()`

This function fixes the first input series as the benchmark and optimizes the zero-cost factor tilt.

**Required Inputs**

- `expected_returns` — Expected returns ordered as the benchmark followed by zero-cost factor portfolios. Example: `expected_returns=expected_returns`.
- `covariance_matrix` — Covariance matrix in the same order as `expected_returns`. Example: `covariance_matrix=covariance_matrix`.

**Optional Inputs**

- `rf` — Default is `0.0`. Risk-free return used in the Sharpe-ratio objective. Example: `rf=0.0`.
- `base_index` — Default is `0`. Identifies the fixed benchmark exposure. Example: `base_index=0`.

**Outputs**

- `tuple` — Returns `(exposures, expected_return, volatility)`. The first exposure is `1.0`; the remaining exposures are optimized factor tilts. Example: `active_exposures, active_return, active_volatility = fm.factor_tilt_portfolio(...)`.

In [ ]:
active_exposures, active_return, active_volatility = fm.factor_tilt_portfolio(
    expected_returns,
    covariance_matrix,
    rf=0.0,
    base_index=0,
)

print(active_exposures)
print(active_return)
print(active_volatility)

### Test Trading Strategy

In [ ]:
# For comparison with Ex07-Ken_French.ipynb
#tangent_weights = np.array([1, 0.250])

# Create realized excess portfolio returns
excess = df[['mkt-rf', 'dec1_dec10']].to_numpy()   # shape (T,2)

# Calculate the excess returns as a 1D array
# @ performs an inner product
excess_array = excess @ active_exposures

# Convert the Series to a DataFrame
excess = pd.DataFrame(excess_array, index=df.index, columns=['excess'])

# compute total return
total_return = excess['excess'] + df['rf']

# Get monthly mean returns, volatility, and sharpe
# Call .mean() to get a scalar mean from the total_return Series
mean_monthly = total_return.mean()
# Select the 'excess' column before calling .mean() as 'excess' is a DataFrame
mean_excess_monthly  = excess['excess'].mean()
# Select the 'excess' column before calling .std() as 'excess' is a DataFrame
std_monthly  = total_return.std()
sharpe_monthly = mean_excess_monthly / std_monthly

# annualize
ann_return = 12 * mean_monthly
ann_vol    = np.sqrt(12) * std_monthly
ann_sharpe = sharpe_monthly * np.sqrt(12)

# Print results
print(f"Average Return: {ann_return:.3f}")
print(f"Annualized Volatility: {ann_vol:.3f}")
print(f"Annualized Sharpe Ratio: {ann_sharpe:.3f}")